# Uyghur (uig) — Arabic Script NLP Pipeline

Uyghur is written in Perso-Arabic script (right-to-left). TurkicNLP provides Arabic-script tokenisation, Apertium FST morphology (Beta quality), full Stanza neural pipeline via the UDT treebank, and bidirectional Arabic↔Latin (ULY) transliteration.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('uig')

## 2. Script Detection — Arabic Script

In [ ]:
from turkicnlp.scripts.detector import detect_script

# Uyghur in Perso-Arabic script
arab_text = "مەن مەكتەپكە بارىمەن."
print("Detected:", detect_script(arab_text))

## 3. Arabic ↔ Latin (ULY) Transliteration

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

arab = "مەن مەكتەپكە بارىمەن."

# Arabic -> Latin (ULY — Uyghur Latin Yéziqi standard)
t = Transliterator("uig", source=Script.ARABIC, target=Script.LATIN)
uly = t.transliterate(arab)
print("ULY:", uly)

# Latin -> Arabic
t_back = Transliterator("uig", source=Script.LATIN, target=Script.ARABIC)
print("Back:", t_back.transliterate(uly))

## 4. Morphological Analysis (Apertium FST — Beta)

In [ ]:
nlp_morph = Pipeline(
    "uig",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
    script="Arab",
)
doc = nlp_morph("مەن مەكتەپكە بارىمەن.")
for w in doc.words:
    print(f"{w.text:<20} lemma={w.lemma:<15} feats={w.feats}")

## 5. POS Tagging, Lemmatisation, and Dependency Parsing (UDT treebank)

In [ ]:
nlp_parse = Pipeline(
    "uig",
    processors=["tokenize", "pos", "lemma", "depparse"],
    script="Arab",
)
doc = nlp_parse("ئۈرۈمچى شىنجاڭنىڭ پايتەختى.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<15} {'Deprel'}")
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<15} {w.deprel}")
print("\nCoNLL-U:\n", doc.to_conllu())

## 6. Translation

In [ ]:
turkicnlp.download("uig", processors=["translate"])
trans = Pipeline("uig", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("شىنجاڭ ئۇيغۇر ئاپتونوم رايونى.")
print("EN:", doc.translation)